# Day 1 - Laboratory Exercise: Predicting Wine Quality

The walkthrough used 344 penguins, which were clean, small, and well behaved.
This laboratory exercise uses 6497 wines, which are none of those things. The
same sequence of steps applies, but here several of them catch a problem that
would otherwise travel silently into your conclusions.

Each wine has eleven physical and chemical measurements and a quality score
from zero to ten, awarded by the median judgement of at least three wine
tasters. Red and white wines are both present and are marked by a `colour`
column.

## How To Work Through This?

There are seven tasks. Each one states a goal, gives you a cell to write in
that is marked `# YOUR CODE HERE`, and follows it with a check cell. Run the
check cell after you write your answer. It either prints a confirmation or
tells you what is wrong, so you never have to wonder whether you may move on.

Do not skip the questions in the markdown between tasks. Several of them are
the point of the exercise, and the code is only there to produce the evidence.

If you finish early, the final section suggests extensions.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

DATA_DIR = Path.cwd().parent / "data"
RANDOM_STATE = 42

sns.set_theme(style="whitegrid")

def check(condition, success, failure):
    """Report whether a task was completed correctly."""
    print(success if condition else failure)

## Task 1: Load The Data And Look At It

Read `wine_quality.csv` from `DATA_DIR` into a variable called `wine`. Then
answer, for yourself, the questions the walkthrough insisted on. How many rows
are there, are any values missing, and what does `describe` say about the
minimum and maximum of each column.

In [ ]:
# YOUR CODE HERE
# Read the file into a DataFrame called `wine`, then inspect it.

In [ ]:
check(
    "wine" in dir() and len(wine) == 6497 and "quality" in wine.columns,
    "Task 1 complete. 6497 wines loaded.",
    "Not right yet. Expected a DataFrame called `wine` with 6497 rows and a `quality` column.",
)

## Task 2: Find The Duplicate Rows

The penguin data had one row per bird. This file makes no such promise, because
two different wines with the same eleven measurements and the same score are
recorded as two identical rows.

Count how many rows are exact duplicates of an earlier row. Use
`DataFrame.duplicated`.

In [ ]:
# YOUR CODE HERE
# Store the count in a variable called `n_duplicates`.

In [ ]:
check(
    "n_duplicates" in dir() and n_duplicates == 1177,
    "Task 2 complete. There are 1177 duplicate rows, which is 18% of the file.",
    "Not right yet. Try wine.duplicated().sum().",
)

Hold on to that number. We return to it in Task 6, where it turns out to change
the headline result of the whole laboratory exercise.

## Task 3: Explore The Target

Produce two things. First, the count of wines at each quality score. Second, a
plot of how quality relates to the measurements, or at least to the one you
suspect matters most.

Before you plot, look at the correlation of every numeric column with
`quality`, sorted. One feature stands out well ahead of the others.

In [ ]:
# YOUR CODE HERE
# Count the wines at each quality score, then build a Series called
# `correlations` holding each numeric column's correlation with quality.

In [ ]:
check(
    "correlations" in dir() and correlations.abs().idxmax() == "alcohol",
    "Task 3 complete. Alcohol correlates most strongly with quality, at about +0.44.",
    "Not right yet. Build a Series called `correlations` of each numeric column against quality.",
)

Two observations are worth making before you continue.

The quality scores are heavily concentrated on 5 and 6, with only 30 wines at
the bottom of the scale and 5 at the top. Any model will see very few examples
of the extremes.

Alcohol correlates with quality at about 0.44, which is the strongest
relationship in the table. Recall Simpson's paradox from the walkthrough, and
notice that this dataset has an obvious grouping variable in `colour`. Checking
whether that correlation survives inside each colour is one of the extensions
at the end.

## Task 4: Predict Quality With Linear Regression

Treat `quality` as a number and fit a linear regression to predict it from the
eleven measurements. Exclude `colour`, which is text, and of course exclude
`quality` itself.

Split the data first, holding back 20 percent with `random_state=RANDOM_STATE`.
Report mean absolute error and R squared, and print the same two numbers for a
`DummyRegressor` that always predicts the training mean.

In [ ]:
# YOUR CODE HERE
# Fit a LinearRegression and compare it against a DummyRegressor baseline.
# Store the linear model's R squared on the test set in `linear_r2`.

In [ ]:
check(
    "linear_r2" in dir() and 0.20 < linear_r2 < 0.32,
    "Task 4 complete. Linear regression reaches an R squared of about 0.26.",
    "Not right yet. Expected an R squared between 0.20 and 0.32 on the held-out wines.",
)

The model beats the baseline, but only just, and an R squared of 0.26 means it
accounts for about a quarter of the variation in quality. Eleven chemical
measurements do not determine what a panel of tasters thinks of a wine. That is
a real finding rather than a failure of the method, and reporting it plainly is
better practice than hunting for a model that appears to do better.

## Task 5: Turn It Into A Classification Problem And Watch Accuracy Mislead You

Define a good wine as one scoring 7 or above, and build a binary target called
`is_good`. Then fit a logistic regression to predict it.

Remember two things from the walkthrough. Logistic regression needs its
features standardised, which you should do inside a `Pipeline` rather than by
hand. And `stratify` the split so both parts hold the same proportion of good
wines.

Print the accuracy. Then print `classification_report`, and print the accuracy
of a `DummyClassifier` using `strategy="most_frequent"`.

In [ ]:
# YOUR CODE HERE
# Build `is_good`, fit a scaled logistic regression, and compare its accuracy
# against a classifier that always predicts the majority class.
# Store the logistic regression's test accuracy in `logistic_accuracy`.

In [ ]:
check(
    "logistic_accuracy" in dir() and 0.78 < logistic_accuracy < 0.86,
    "Task 5 complete. Now read the recall for the 'good' class before you continue.",
    "Not right yet. Expected an accuracy between 0.78 and 0.86.",
)

### Stop And Read The Report

The logistic regression scores about 0.82 accuracy. A model that ignores every
measurement and calls every wine not good scores about 0.80. The difference
between doing real work and doing nothing at all is two percentage points.

Now look at the recall on the good class, which is about 0.26. Of every four
genuinely good wines, the model finds one and misses three. Accuracy hid that
completely, because 80 percent of the wines are not good and getting those
right is enough to look respectable.

This is why the walkthrough insisted on a baseline. Whenever one class is much
larger than the other, accuracy measures the size of the majority class more
than it measures the model.

## Task 6: Random Forests And An Honest Estimate

Fit a `RandomForestClassifier` with 300 trees on the same task and print its
accuracy, precision, recall, and F1 for the good class.

Then evaluate it a second way, with 5-fold cross validation scored on F1. Use
`StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)`. The
shuffle matters here: this file has all the red wines first and all the whites
after them, so folds taken without shuffling are not representative.

In [ ]:
# YOUR CODE HERE
# Fit the forest, report its scores on the held-out set, then cross validate it.
# Store the held-out F1 in `forest_f1` and the cross validated scores in `cv_f1`.

In [ ]:
check(
    "forest_f1" in dir() and forest_f1 > 0.55,
    "Task 6 complete. The forest reaches an F1 near 0.66, far above the logistic regression's 0.37.",
    "Not right yet. Expected an F1 above 0.55 for the random forest.",
)

### The Result Is Too Good And Task 2 Explains Why

The forest reports an F1 around 0.66 both on the held-out set and under cross
validation. Two independent estimates agreeing is normally reassuring.

They agree because they share the same flaw. Recall the 1177 duplicate rows you
counted in Task 2. A random split puts some copies of a duplicated wine in the
training set and its twin in the test set. The forest, which is perfectly
capable of memorising individual rows, then recognises a wine it has already
seen. That is not prediction, it is recall of the training data, and it
inflates every score computed this way.

Cross validation did not save us, because it splits randomly too.

## Task 7: Measure It Again Without The Leak

Remove the duplicate rows with `drop_duplicates`, then repeat both evaluations
from Task 6 on the deduplicated data. Report the held-out F1 and the cross
validated F1.

In [ ]:
# YOUR CODE HERE
# Deduplicate, then rerun both the held-out and the cross validated evaluation.
# Store the deduplicated held-out F1 in `honest_f1`.

In [ ]:
check(
    "honest_f1" in dir() and 0.40 < honest_f1 < 0.55,
    "Task 7 complete. The honest F1 is near 0.47, about seven tenths of what the leak suggested.",
    "Not right yet. Expected a held-out F1 between 0.40 and 0.55 after deduplicating.",
)

### What Just Happened

The model did not change. The data did not change in any way that a chemist
would recognise. Removing rows that were counted twice dropped the F1 from
about 0.66 to about 0.47.

Had you reported the first number, you would have overstated the model's
performance by about thirty percent. Nothing in the modelling code was wrong.
The error was made in Task 2, at the moment the duplicates were counted and
then ignored.

This is the most important thing in today's session. Most published failures of
applied machine learning are not exotic. They are a train and test set that
were not as separate as somebody assumed.

## Extensions

Work on these in any order if you have time left.

1. Check Simpson's paradox on this data. Compute the correlation between
   alcohol and quality overall, then again separately for red and for white.
   Does the relationship hold up inside each colour?
2. The good class is only 19 percent of the data. Refit the forest with
   `class_weight="balanced"` and see what it does to precision and recall.
   Which of the two improves, and which gets worse?
3. Instead of using a threshold of 0.5 on the forest's probabilities, sweep the
   threshold and plot precision and recall against it. Pick the threshold you
   would choose if the goal were to shortlist wines for a human to taste, and
   justify it in a sentence.
4. Add `colour` as a feature, encoded with `get_dummies`. Does knowing whether
   a wine is red or white help predict its quality?
5. Print the forest's `feature_importances_`. Compare the ranking against the
   correlations you computed in Task 3, and explain any feature that ranks high
   in one and low in the other.

## What To Take Away

- Count the duplicate rows before you split. A random split over duplicated
  data leaks the test set into training, and both the held-out score and cross
  validation will agree with each other while both being wrong.
- Print a baseline next to every score. On imbalanced data, accuracy mostly
  measures the size of the majority class.
- Read precision and recall separately, and decide which one your application
  actually cares about before you look at either.
- Shuffle before cross validating whenever the rows arrived in a meaningful
  order.
- A model that explains a quarter of the variation, reported honestly, is worth
  more than one that appears to explain two thirds because of a leak.